# Train and export the routers — 4-stage IDK cascade (ImageNetV2)

**Run this notebook on Colab (or any PC with a GPU) — not on the Jetson.** It trains every router the Jetson script needs and saves them to disk with `joblib`, exactly like the previous `train_routers` notebook. Nothing is evaluated here: accuracy, latency, router overhead and WCET must be measured **on the Jetson** with `idk_cascade_4stage_jetson_nano.py`.

**Cascade.** `A` ResNet-18 → `B` ResNet-34 → `C` ResNet-50 → `D` ResNet-152 (ImageNet-pretrained, torchvision `IMAGENET1K_V1`), data = ImageNetV2-TopImages (10,000 images).

**Routers** (escalate-or-stop). After stage `A`, `B` or `C`: if its confidence is ≥ τ (0.90) the answer is accepted and no router is called; otherwise a router decides **escalate** (run the next stage) or **stop** (keep the current answer). Three routers per family — `R_AB`, `R_BC`, `R_CD` — each trained on **all train-split images**, on the 3 features (`confidence`, `entropy`, `margin`) of the stage that just ran, with label **current stage wrong ∧ next stage right**. Five families: XGBoost, Random Forest, Logistic Regression, Decision Tree, GA-optimised linear heuristic (all with the same class weighting, hyper-parameters and decision threshold θ = 0.20).

**What is exported** (folder `exported_routers/`):

| File | Content |
|---|---|
| `bench_<family>.joblib` | 5 bench routers (= `R_AB` of the first dataset and first seed), used on the Jetson to time one router call |
| `<dataset>_<seed>_<family>_<node>.joblib` | the routers: 8 seeds × 5 families × 3 nodes (`ab`, `bc`, `cd`) per dataset |
| `manifest.json` | seeds, split size, θ, dataset fingerprint (image list + order), package versions |
| `cumulative_compute_time.csv` | router fit time per family |

The train/test split of every seed is `train_test_split(np.arange(10000), test_size=0.2, random_state=seed)`; the Jetson script reproduces it, and refuses to run if its ImageNetV2 image list differs from the one used here (fingerprint check).

**Then:** the last cell zips the folder and downloads it (on Colab). Unzip it on the Jetson (`unzip exported_routers.zip -d exported_routers`) and run

```
python3 idk_cascade_4stage_jetson_nano.py --router_dir ./exported_routers --output_dir ./results
```

In [ ]:
!pip install -q scikit-learn==1.9.0 xgboost==3.2.0

In [ ]:
import os, sys, json, time, shutil, tarfile, hashlib, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib
from PIL import Image
import sklearn, xgboost
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 50)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('numpy', np.__version__, '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__, '| xgboost', xgboost.__version__)
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', device)

## 1. Configuration

In [ ]:
# ============================== EXPORT CONFIGURATION ==============================
RUN_DATASETS = ['TopImages']                # the 4-stage paper's main set; optional: 'Threshold0.7', 'MatchedFrequency'
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49]    # one set of routers per seed; seed = random_state of the 80/20 split
TEST_SIZE = 0.20
THETA = 0.20                                # decision threshold: escalate if P(escalate) >= THETA (used by the GA fitness)
DEFAULT_TAU = 0.90                          # confidence threshold of the study (only recorded in the manifest; used at evaluation)

ROUTER_DIR = './exported_routers'
DATA_ROOT = './data_imagenetv2'
DELETE_ARCHIVE_AFTER_EXTRACT = True
BATCH_SIZE, NUM_WORKERS = 64, 2             # telemetry pass (lower BATCH_SIZE if the GPU runs out of memory)

# Latency (ms) of the stage each router GATES - only used in the cost-sensitive class weight  clip(neg/pos / relative_cost, 1, 10), where
# relative_cost = gated-stage latency / cheapest gated-stage latency (only the RATIOS matter).
# None = measure on THIS machine. To match the Jetson, paste the Mean_ms values of its model_latency.csv, e.g. {'B': 9.1, 'C': 14.7, 'D': 41.3}
TARGET_LATENCY_MS = {'B': None, 'C': None, 'D': None}
LAT_REPS, LAT_WARMUP = 100, 20

STAGES       = ['A', 'B', 'C', 'D']
STAGE_MODELS = ['resnet18', 'resnet34', 'resnet50', 'resnet152']

## 2. Data: ImageNetV2-TopImages

Downloaded from the ImageNetV2 Hugging Face mirror (~1.2 GB; the legacy S3 bucket is only a fallback). Class folders are named by the integer class index, so the label is read from the folder name (`ImageFolder` would sort `"0","1","10",…` as strings and mis-assign labels).

In [ ]:
HF = 'https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/'       # primary mirror (the S3 bucket now returns 403)
S3 = 'https://s3-us-west-2.amazonaws.com/imagenetv2public/'                      # legacy fallback
DATASET_SPECS = {
    'TopImages':        dict(archive='imagenetv2-top-images.tar.gz',        folder='imagenetv2-top-images-format-val'),
    'Threshold0.7':     dict(archive='imagenetv2-threshold0.7.tar.gz',      folder='imagenetv2-threshold0.7-format-val'),
    'MatchedFrequency': dict(archive='imagenetv2-matched-frequency.tar.gz', folder='imagenetv2-matched-frequency-format-val'),
}
for _spec in DATASET_SPECS.values():
    _spec['urls'] = [HF + _spec['archive'], S3 + _spec['archive']]

# Standard ImageNet evaluation preprocessing (what the torchvision weights were trained/evaluated with)
IMAGENET_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                         T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

class ImageNetV2Folder(Dataset):
    """root/<class_idx>/<image files>; label = int(<class_idx>). Deterministic (sorted) order."""
    def __init__(self, root, transform):
        self.root = root
        self.transform = transform
        self.samples = []
        class_dirs = sorted((d for d in os.listdir(root) if d.isdigit()), key=int)
        for d in class_dirs:
            cdir = os.path.join(root, d)
            for f in sorted(os.listdir(cdir)):
                p = os.path.join(cdir, f)
                if not f.startswith('.') and os.path.isfile(p):
                    self.samples.append((p, int(d)))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as im:
            img = im.convert('RGB')
        return self.transform(img), label

def dataset_fingerprint(ds):
    """Identity of the image list + order. The training notebook and the Jetson script must see the SAME one,
    because the train/test split indices refer to positions in this list."""
    h = hashlib.sha1()
    for p, lbl in ds.samples:
        h.update('{}:{}\n'.format(os.path.relpath(p, ds.root).replace(os.sep, '/'), lbl).encode('utf-8'))
    return {'n_images': len(ds), 'sha1': h.hexdigest()}

def download_file(url, dest):
    """Streaming download with a browser-like User-Agent and a progress line every 5%."""
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (idk-cascade-notebook)'})
    with urllib.request.urlopen(req, timeout=60) as resp, open(dest, 'wb') as out:
        total = int(resp.headers.get('Content-Length') or 0)
        done, last = 0, -1
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            out.write(chunk)
            done += len(chunk)
            if total and (done * 100 // total) // 5 != last:
                last = (done * 100 // total) // 5
                print('  {:3d}%  ({:.0f} MB)'.format(done * 100 // total, done / 1e6), flush=True)
    if total and done != total:
        raise IOError('Incomplete download ({} of {} bytes)'.format(done, total))

def find_dataset_root(base):
    """The class-folder directory (numeric folders 0, 1, ...) inside an extracted archive, wherever it is wrapped."""
    found = []
    for dirpath, dirnames, _ in os.walk(base):
        digit_dirs = [d for d in dirnames if d.isdigit()]       # ignore stray folders (PaxHeader, __MACOSX, ...)
        if '0' in digit_dirs and len(digit_dirs) >= 10:
            found.append(dirpath)
    if len(found) != 1:
        raise RuntimeError('Could not identify a unique class-folder root in the archive ({} candidates).'.format(len(found)))
    return found[0]

def prepare_dataset(name):
    spec = DATASET_SPECS[name]
    os.makedirs(DATA_ROOT, exist_ok=True)
    folder = os.path.join(DATA_ROOT, spec['folder'])
    if not os.path.isdir(folder):
        archive = os.path.join(DATA_ROOT, spec['archive'])
        if not os.path.isfile(archive):
            errors = []
            for url in spec['urls']:
                print('Downloading {} (~1.2 GB) from {}'.format(name, url), flush=True)
                try:
                    download_file(url, archive + '.part')
                    os.replace(archive + '.part', archive)     # only a complete download gets the final name
                    break
                except Exception as e:                          # HTTP 403/404, network error, ...
                    errors.append('{}: {}'.format(url, e))
                    print('  failed ({}) - trying next mirror'.format(e), flush=True)
                    if os.path.exists(archive + '.part'):
                        os.remove(archive + '.part')
            else:
                raise RuntimeError('All download mirrors failed:\n  ' + '\n  '.join(errors) +
                                   '\nDownload {} manually (e.g. from https://huggingface.co/datasets/vaishaal/ImageNetV2) '
                                   'and put it in {} - or extract it there so that {} exists.'.format(
                                       spec['archive'], os.path.abspath(DATA_ROOT), folder))
        print('Extracting {} ...'.format(spec['archive']), flush=True)
        tmp = os.path.join(DATA_ROOT, '_extract_tmp')
        shutil.rmtree(tmp, ignore_errors=True)
        os.makedirs(tmp)
        with tarfile.open(archive, 'r:*') as tf:                   # auto-detects gzip / bz2 / xz / plain tar
            try:
                tf.extractall(tmp, filter='data')
            except TypeError:                                   # older Python without extraction filters
                tf.extractall(tmp)
        os.replace(find_dataset_root(tmp), folder)             # a partially extracted folder is never left behind
        shutil.rmtree(tmp, ignore_errors=True)
        if DELETE_ARCHIVE_AFTER_EXTRACT:
            os.remove(archive)
    ds = ImageNetV2Folder(folder, IMAGENET_TF)
    n_cls = len(set(lbl for _, lbl in ds.samples))
    print('{}: {} images, {} classes'.format(name, len(ds), n_cls))
    if len(ds) != 10000 or n_cls != 1000:
        print('  WARNING: expected 10,000 images in 1,000 classes - check the extracted folder.')
    return ds

DATASETS_OBJ = {name: prepare_dataset(name) for name in RUN_DATASETS}

## 3. Models, stage functions and telemetry pass

Every image goes once through all four models. For each stage we store the router features (`confidence`, `entropy`, `margin`), the prediction and its correctness. These are the training data of the routers.

In [ ]:
WEIGHT_ENUMS = {'resnet18': 'ResNet18_Weights', 'resnet34': 'ResNet34_Weights',
                'resnet50': 'ResNet50_Weights', 'resnet152': 'ResNet152_Weights'}

def load_imagenet_resnet(arch):
    """torchvision ResNet with the original IMAGENET1K_V1 weights, in eval mode on the execution device."""
    ctor = getattr(torchvision.models, arch)
    enum = getattr(torchvision.models, WEIGHT_ENUMS[arch], None)
    if enum is not None:
        model = ctor(weights=enum.IMAGENET1K_V1)
    else:                                                    # older torchvision (e.g. Jetson JetPack builds)
        model = ctor(pretrained=True)
    return model.to(device).eval()

@torch.no_grad()
def stage_with_features(model, x):
    """Stages A/B/C: returns an (n,4) float32 array [confidence, entropy, margin, predicted class]."""
    probs = torch.softmax(model(x), dim=1)
    conf, pred = probs.max(dim=1)
    ent = -(probs * torch.log(probs + 1e-6)).sum(dim=1)
    top2 = torch.topk(probs, k=2, dim=1).values
    margin = top2[:, 0] - top2[:, 1]
    return torch.stack([conf, ent, margin, pred.to(probs.dtype)], dim=1).cpu().numpy()

@torch.no_grad()
def stage_final(model, x):
    """Stage D (terminal expert): argmax only."""
    return model(x).argmax(dim=1).cpu().numpy()

def to_device_input(ds, i):
    """One preprocessed image as a (1,3,224,224) tensor on the execution device."""
    return ds[i][0].unsqueeze(0).to(device)

def time_stage(fn, model, x, n_reps, n_warmup):
    """Wall-clock time (ms) of n_reps single-input calls of fn(model, x)."""
    use_cuda = device.type == 'cuda'
    for _ in range(n_warmup):
        fn(model, x)
    if use_cuda:
        torch.cuda.synchronize()
    ts = np.empty(n_reps)
    for i in range(n_reps):
        t0 = time.perf_counter()
        fn(model, x)
        if use_cuda:
            torch.cuda.synchronize()
        ts[i] = (time.perf_counter() - t0) * 1000.0
    return ts

def timing_stats(ts):
    ts = np.asarray(ts, dtype=float)
    return dict(mean=float(ts.mean()), std=float(ts.std(ddof=1)) if len(ts) > 1 else 0.0,
                p99=float(np.percentile(ts, 99)), max=float(ts.max()))

def profile_stages(models, probe_x, n_reps, n_warmup):
    """Single-input (batch 1) timing of every stage. Returns a list of 4 timing_stats dicts."""
    fns = [stage_with_features, stage_with_features, stage_with_features, stage_final]
    return [timing_stats(time_stage(fns[k], models[k], probe_x, n_reps, n_warmup)) for k in range(4)]

def profile_dataset(name, ds, models):
    """One pass of all 4 models over the whole dataset. Returns arrays indexed [stage, image, ...]."""
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == 'cuda'))
    out = [[] for _ in range(4)]
    labels, n_done, t0 = [], 0, time.perf_counter()
    for bi, (imgs, labs) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        for k in range(4):
            out[k].append(stage_with_features(models[k], imgs))
        labels.append(labs.numpy())
        n_done += len(labs)
        if (bi + 1) % 20 == 0 or n_done == len(ds):
            el = time.perf_counter() - t0
            print('  {}: {}/{} images | {:.0f}s elapsed | ETA {:.0f}s'.format(
                name, n_done, len(ds), el, el / n_done * (len(ds) - n_done)), flush=True)
    raw = np.stack([np.concatenate(o) for o in out]).astype(np.float64)       # (4, N, 4)
    labels = np.concatenate(labels)
    pred = raw[:, :, 3].astype(np.int64)
    feat = np.ascontiguousarray(raw[:, :, :3])                                # (4, N, 3): conf, entropy, margin
    return dict(feat=feat, pred=pred, correct=(pred == labels[None, :]), labels=labels)

NODES = ('AB', 'BC', 'CD')
NODE_STAGE = {'AB': 0, 'BC': 1, 'CD': 2}          # router R_xy reads the features of stage x and gates stage y

def subset(tel, idx):
    return dict(feat=tel['feat'][:, idx, :], conf=tel['feat'][:, idx, 0], correct=tel['correct'][:, idx])

def node_dataset(trn, node):
    """Training data of one router: ALL train-split images, features of the stage that just ran.
    Label 1 = escalating would fix an error (current stage wrong AND next stage right)."""
    s = NODE_STAGE[node]
    y = (~trn['correct'][s]) & trn['correct'][s + 1]
    return trn['feat'][s], y.astype(int)

def router_filename(dname, seed, family, node):
    """File name of one exported router, e.g.  TopImages_42_rf_ab.joblib"""
    return '{}_{}_{}_{}.joblib'.format(dname, seed, family, node.lower())

In [ ]:
MODELS = [load_imagenet_resnet(a) for a in STAGE_MODELS]

# latency of the gated stages (class weights only)
if any(v is None for v in TARGET_LATENCY_MS.values()):
    print('Measuring stage latencies on this machine (class-weight input only) ...')
    _st = profile_stages(MODELS, to_device_input(DATASETS_OBJ[RUN_DATASETS[0]], 0), LAT_REPS, LAT_WARMUP)
    _measured = {'B': _st[1]['mean'], 'C': _st[2]['mean'], 'D': _st[3]['mean']}
else:
    _measured = {}
LAT_USED = {k: (TARGET_LATENCY_MS[k] if TARGET_LATENCY_MS[k] is not None else _measured[k]) for k in ('B', 'C', 'D')}
GATED_LAT_MS = {'AB': LAT_USED['B'], 'BC': LAT_USED['C'], 'CD': LAT_USED['D']}
# Class weights use the gated stage's cost RELATIVE to the cheapest gated stage (B = 1.0). With raw milliseconds, 1/latency is far
# below 1, so neg/pos x 1/latency was clipped to its floor of 1.0 at every node and the weighting did nothing.
_cheapest = min(GATED_LAT_MS.values())
GATED_LAT = {k: v / _cheapest for k, v in GATED_LAT_MS.items()}
print('Gated-stage latency (ms):', {k: round(v, 3) for k, v in GATED_LAT_MS.items()})
print('Relative cost used for the class weights (cheapest gated stage = 1):', {k: round(v, 3) for k, v in GATED_LAT.items()})

TEL = {}
for name in RUN_DATASETS:
    print('Telemetry pass: {}'.format(name))
    TEL[name] = profile_dataset(name, DATASETS_OBJ[name], MODELS)
    t = TEL[name]
    print('  Top-1 accuracy per stage (%):  ' + '   '.join('{} {:.2f}'.format(s, t['correct'][k].mean() * 100) for k, s in enumerate(STAGES)))
    if t['correct'][3].mean() < 0.30:
        raise RuntimeError('ResNet-152 accuracy is implausibly low - labels/data are probably wrong.')

## 4. Router definitions

Same hyper-parameters as the previous study: XGBoost / Random Forest 50 trees, depth 3, 2 threads; Decision Tree depth 3; Logistic Regression `max_iter=1000`; GA heuristic — linear rule $s=w_{conf}\,conf+w_{ent}\,entropy+w_{margin}\,margin+b$, $P=1/(1+e^{-s})$, population 40, 60 generations, tournament size 2, elitism, uniform crossover, Gaussian mutation (σ = 0.6, truncated to [−8, 8], 20% gene rate), 25 random restarts, fitness = weighted accuracy at θ. The `HeuristicRouter` class is pickled **by name**: the Jetson script defines the identical class.

In [ ]:
FAMILIES = ['xgb', 'rf', 'logreg', 'dtree', 'heuristic']
FAMILY_LABELS = {'xgb': 'XGBoost', 'rf': 'Random Forest', 'logreg': 'Logistic Regression',
                 'dtree': 'Decision Tree', 'heuristic': 'GA Heuristic'}

def _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback=1.0):
    # latency_gated = cost of the gated stage RELATIVE to the cheapest gated stage (dimensionless, >= 1); weight = neg/pos / cost, clipped to [1, 10]
    asymmetric_multiplier = cost_missed_fallback / max(latency_gated, 1e-6)
    num_pos = np.sum(y_train == 1)
    num_neg = np.sum(y_train == 0)
    base_weight = num_neg / max(num_pos, 1)
    return float(np.clip(base_weight * asymmetric_multiplier, 1.0, 10.0))

def train_xgb_router(X_train, y_train, latency_gated, cost_missed_fallback=1.0, seed=42):
    w = _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback)
    r = xgb.XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, scale_pos_weight=w,
                          eval_metric='logloss', random_state=seed, n_jobs=2)
    r.fit(X_train, y_train)
    return r

def train_rf_router(X_train, y_train, latency_gated, cost_missed_fallback=1.0, seed=42):
    w = _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback)
    r = RandomForestClassifier(n_estimators=50, max_depth=3, class_weight={0: 1.0, 1: w},
                               random_state=seed, n_jobs=2)
    r.fit(X_train, y_train)
    return r

def train_logreg_router(X_train, y_train, latency_gated, cost_missed_fallback=1.0, seed=42):
    w = _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback)
    r = LogisticRegression(class_weight={0: 1.0, 1: w}, max_iter=1000, random_state=seed)
    r.fit(X_train, y_train)
    return r

def train_dtree_router(X_train, y_train, latency_gated, cost_missed_fallback=1.0, seed=42):
    w = _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback)
    r = DecisionTreeClassifier(max_depth=3, class_weight={0: 1.0, 1: w}, random_state=seed)
    r.fit(X_train, y_train)
    return r

# ---- GA-optimised linear heuristic -------------------------------------------------------------
def _weighted_accuracy_fitness(weights, X_arr, y_arr, sample_weights, theta=0.20):
    score = X_arr @ weights[:3] + weights[3]
    logit_theta = np.log(theta / (1 - theta))
    preds = (score >= logit_theta).astype(int)
    correct = (preds == y_arr).astype(float)
    return np.sum(correct * sample_weights) / np.sum(sample_weights)

def _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=42, pop_size=40, generations=60,
                              gene_range=8.0, mutation_rate=0.2, mutation_scale=0.6, theta=0.20):
    rng = np.random.default_rng(seed)
    population = rng.uniform(-gene_range, gene_range, size=(pop_size, 4))
    best_individual, best_fitness = None, -np.inf
    for _ in range(generations):
        fitnesses = np.array([_weighted_accuracy_fitness(ind, X_arr, y_arr, sample_weights, theta=theta)
                              for ind in population])
        gen_best_idx = np.argmax(fitnesses)
        if fitnesses[gen_best_idx] > best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_individual = population[gen_best_idx].copy()
        new_population = [best_individual.copy()]                         # elitism
        while len(new_population) < pop_size:
            i, j = rng.integers(0, pop_size, size=2)                      # tournament (size 2)
            p1 = population[i] if fitnesses[i] > fitnesses[j] else population[j]
            i, j = rng.integers(0, pop_size, size=2)
            p2 = population[i] if fitnesses[i] > fitnesses[j] else population[j]
            mask = rng.random(4) < 0.5                                    # uniform crossover
            child = np.where(mask, p1, p2)
            mutate_mask = rng.random(4) < mutation_rate                   # Gaussian mutation
            child = child + mutate_mask * rng.normal(0, mutation_scale, size=4)
            child = np.clip(child, -gene_range, gene_range)
            new_population.append(child)
        population = np.array(new_population[:pop_size])
    return best_individual, best_fitness

class HeuristicRouter:
    """Linear score + sigmoid, evaluated directly with array operations (no library dispatch)."""
    def __init__(self, weights):
        self.weights = np.asarray(weights, dtype=float)
    def predict_proba(self, X):
        X_arr = X.values if hasattr(X, 'values') else np.asarray(X)
        score = X_arr @ self.weights[:3] + self.weights[3]
        prob = 1.0 / (1.0 + np.exp(-score))
        return np.column_stack([1.0 - prob, prob])

def train_heuristic_router(X_train, y_train, latency_gated, cost_missed_fallback=1.0, seed=42, n_restarts=25):
    w = _cost_sensitive_pos_weight(y_train, latency_gated, cost_missed_fallback)
    X_arr = np.asarray(X_train, dtype=float)
    y_arr = np.asarray(y_train)
    sample_weights = np.where(y_arr == 1, w, 1.0)
    best_w, best_f = None, -np.inf
    for restart in range(n_restarts):
        weights, fit = _genetic_algorithm_search(X_arr, y_arr, sample_weights, seed=seed * 1000 + restart, theta=THETA)
        if fit > best_f:
            best_f, best_w = fit, weights
    return HeuristicRouter(best_w)

class ConstantRouter:
    """Fallback if a node's training labels contain a single class (cannot happen with realistic data)."""
    def __init__(self, p):
        self.p = float(p)
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.full(n, 1.0 - self.p), np.full(n, self.p)])

TRAIN_FNS = {'xgb': train_xgb_router, 'rf': train_rf_router, 'logreg': train_logreg_router,
             'dtree': train_dtree_router, 'heuristic': train_heuristic_router}

TRAIN_TIME = {f: 0.0 for f in FAMILIES}          # cumulative fit time per family (for the compute-time table)

def timed_train(fam, X, y, latency_gated, seed):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        print('  WARNING: single-class training labels for {} - using a constant router.'.format(FAMILY_LABELS[fam]))
        return ConstantRouter(float(y[0]) if len(y) else 1.0)
    t0 = time.perf_counter()
    router = TRAIN_FNS[fam](np.asarray(X, dtype=float), y, latency_gated, seed=seed)
    TRAIN_TIME[fam] += time.perf_counter() - t0
    return router

## 5. Train and export

For every dataset, seed and node the training data are the **whole train split** (features of the stage that just ran, label = current stage wrong ∧ next stage right). The first printed line shows the class balance and the class weight of each router. The bench router of each family is the `R_AB` router of the first dataset and first seed (saved a second time under `bench_<family>.joblib`).

In [ ]:
def save_router(router, directory, filename):
    joblib.dump(router, os.path.join(directory, filename))

os.makedirs(ROUTER_DIR, exist_ok=True)
if any(f.endswith('.joblib') for f in os.listdir(ROUTER_DIR)):
    print('NOTE: {} already contains .joblib files - files with the same name are overwritten.'.format(ROUTER_DIR))

n_saved, t_start = 0, time.perf_counter()
for dname in RUN_DATASETS:
    tel = TEL[dname]
    N = tel['feat'].shape[1]
    print('Training routers for {} ...'.format(dname), flush=True)
    for si, seed in enumerate(SEEDS):
        tr_idx, _ = train_test_split(np.arange(N), test_size=TEST_SIZE, random_state=seed)
        trn = subset(tel, tr_idx)
        for node in NODES:
            X, y = node_dataset(trn, node)
            if si == 0:
                print('  R_{}: {} training images, {:.1f}% positive (escalating fixes an error), class weight {:.2f}'.format(
                    node, len(y), y.mean() * 100, _cost_sensitive_pos_weight(y, GATED_LAT[node])))
            for fam in FAMILIES:
                router = timed_train(fam, X, y, GATED_LAT[node], seed)
                save_router(router, ROUTER_DIR, router_filename(dname, seed, fam, node)); n_saved += 1
                if dname == RUN_DATASETS[0] and si == 0 and node == 'AB':
                    save_router(router, ROUTER_DIR, 'bench_{}.joblib'.format(fam)); n_saved += 1
        print('  {} seed {} done ({}/{})'.format(dname, seed, si + 1, len(SEEDS)), flush=True)
TOTAL_WALL = time.perf_counter() - t_start
expected = len(RUN_DATASETS) * len(SEEDS) * len(FAMILIES) * len(NODES) + len(FAMILIES)
print('\nSaved {} router files to {} (expected {}).'.format(n_saved, os.path.abspath(ROUTER_DIR), expected))
assert n_saved == expected

In [ ]:
# ---- manifest (read by the Jetson script) ----
manifest = dict(
    format_version=1, study='4-stage IDK cascade router overhead (ImageNetV2)',
    stages=STAGE_MODELS, nodes=[n.lower() for n in NODES], families=FAMILIES,
    seeds=SEEDS, test_size=TEST_SIZE, theta=THETA, default_tau=DEFAULT_TAU,
    router_file_pattern='{dataset}_{seed}_{family}_{node}.joblib', bench_file_pattern='bench_{family}.joblib',
    datasets={name: dict(fingerprint=dataset_fingerprint(DATASETS_OBJ[name])) for name in RUN_DATASETS},
    class_weight_latency_ms={k: float(v) for k, v in GATED_LAT_MS.items()},
    class_weight_relative_cost={k: float(v) for k, v in GATED_LAT.items()},
    class_weight_scheme='clip(neg/pos / relative_cost, 1, 10); relative_cost = gated-stage latency / cheapest gated-stage latency',
    versions={'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__, 'numpy': np.__version__,
              'torch': torch.__version__, 'torchvision': torchvision.__version__},
    trained_on=str(device))
with open(os.path.join(ROUTER_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

# ---- cumulative router compute time (training only) ----
total_fit = sum(TRAIN_TIME.values())
df_compute = pd.DataFrame([{'Router Family': FAMILY_LABELS[k], 'Time (s)': round(TRAIN_TIME[k], 2),
                            'Share (%)': round(TRAIN_TIME[k] / total_fit * 100, 1) if total_fit > 0 else 0.0} for k in FAMILIES])
df_compute.to_csv(os.path.join(ROUTER_DIR, 'cumulative_compute_time.csv'), index=False)
print('=' * 50)
print('CUMULATIVE ROUTER COMPUTE TIME (training)')
print('=' * 50)
display(df_compute.set_index('Router Family'))
print('Total router fit time: {:.2f} s | total wall-clock of the training loop: {:.2f} s ({:.2f} min)'.format(total_fit, TOTAL_WALL, TOTAL_WALL / 60))
print('\nManifest:')
print(json.dumps({k: manifest[k] for k in ('seeds', 'test_size', 'theta', 'datasets', 'class_weight_latency_ms', 'class_weight_relative_cost')}, indent=2))

## 6. Download

Zips the folder and, on Colab, downloads it. Unzip on the Jetson into a folder and pass that folder as `--router_dir`.

In [ ]:
shutil.make_archive('exported_routers', 'zip', ROUTER_DIR)
print('Created exported_routers.zip ({:.1f} MB)'.format(os.path.getsize('exported_routers.zip') / 1e6))
try:
    from google.colab import files
    files.download('exported_routers.zip')
except ImportError:
    print('Not running on Colab - copy exported_routers.zip (or the folder {}) to the Jetson yourself, e.g. with scp.'.format(ROUTER_DIR))